# Notebook 64 — Final Paper Construction

## A Shared Governing Field with a Sparse Symbolic Chart

This notebook is the final, camera-ready construction notebook for `zeta-constraint-lab`.

It consolidates the result from Notebooks 41–63:

> A shared low-order residual field governs dynamics in `(r,c)`, while regime variation is captured by a sparse symbolic coordinate chart for the coefficient vector. The coefficient manifold is low-dimensional and partitioned by forcing/flow/system variables. Generalization across regimes is controlled by a universality boundary where dense models can fail but sparse symbolic structure remains stable.

This notebook intentionally avoids exploratory branching. It produces:

1. Locked governing field.
2. Pruned symbolic chart.
3. Minimal paper figures.
4. Statistical validation.
5. Final summary tables.
6. Exportable figure/table/Markdown artifacts.

## Final model

\[
g(r,c;\beta)
=
\beta_0
+\beta_c c
+\beta_r r
+\beta_{c^3}c^3
+\beta_{r^2}r^2
+\beta_{rc^2}rc^2
\]

\[
\beta
=
f(\text{forcing mode},\text{flow mode},\text{system},\text{task},k)
\]

In [ ]:
# Minimal setup

import warnings
warnings.filterwarnings("ignore")

import os
import glob
import math
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, Ridge
from sklearn.model_selection import KFold, LeaveOneOut
from sklearn.metrics import mean_squared_error, silhouette_score
from sklearn.cluster import KMeans

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda x: x

np.random.seed(42)

OUTPUT_DIR = "paper64_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "axes.grid": True,
    "grid.alpha": 0.25,
})

## 1. Data loading and schema alignment

The notebook auto-detects exported residual-flow data from prior notebooks. If no data file exists in the runtime, it uses the same synthetic fallback family used throughout the later notebooks so the paper-construction notebook remains runnable.

In [ ]:
DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet",
        "*residual*flow*.csv",
        "*governing*flow*.parquet",
        "*governing*flow*.csv",
        "*.parquet",
        "*.csv",
        "*.pkl",
        "*.pickle",
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat))
        candidates.extend(glob.glob(os.path.join("data", pat)))
        candidates.extend(glob.glob(os.path.join("outputs", pat)))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        n_paths = 14
                        n_steps = 42
                        c_grid = np.linspace(-1.25, 1.05, n_steps)

                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.05, 5: 0.02, 7: 0.06}[k]
                        flow_shift = 0.05 if flow_mode == "nonlinear" else -0.02
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(n_paths):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift + flow_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if task == "zeta_vs_poisson":
                                    g -= 0.015 * c
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                delta_c = c_grid[min(window_id + 1, n_steps - 1)] - c if window_id < n_steps - 1 else c - c_grid[max(window_id - 1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * delta_c

                                rows.append({
                                    "system": system,
                                    "task": task,
                                    "forcing_mode": forcing_mode,
                                    "k": k,
                                    "flow_mode": flow_mode,
                                    "condition_coord": c,
                                    "residual": r,
                                    "predicted_flow": g + noise,
                                    "next_residual": next_r,
                                    "delta_condition": delta_c,
                                    "sample_id": sample_id,
                                    "path_id": path_id,
                                    "window_id": window_id,
                                })
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

alias_groups = {
    "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
    "residual": ["residual", "r", "resid"],
    "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
    "next_residual": ["next_residual", "r_next", "next_r"],
    "delta_condition": ["delta_condition", "dc", "delta_c"],
    "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
}

def align_schema(df):
    cols = list(df.columns)
    rename_map = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in cols and a != canonical:
                rename_map[a] = canonical
                break
    df = df.rename(columns=rename_map)

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]

    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        df["delta_condition"] = np.gradient(df["condition_coord"].to_numpy())

    defaults = {
        "system": "unknown_system",
        "task": "unknown_task",
        "forcing_mode": "unknown_forcing",
        "k": 5,
        "flow_mode": "unknown_flow",
        "sample_id": np.arange(len(df)),
        "path_id": 0,
        "window_id": np.arange(len(df)),
    }
    for key, value in defaults.items():
        if key not in df.columns:
            df[key] = value

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")
    return df

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None

if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)

print("Data source:", data_source)
print("Synthetic fallback:", USE_SYNTHETIC_FALLBACK)
print("Rows:", len(df))
display(df.head())

## 2. Locked governing field

This is the core object used throughout the final paper.

In [ ]:
TERM_NAMES = ["1", "c", "r", "c^3", "r^2", "r c^2"]
COEF_COLS = [f"coef_{t}" for t in TERM_NAMES]

def governing_field(r, c, beta):
    beta = np.asarray(beta, dtype=float)
    return (
        beta[0]
        + beta[1] * c
        + beta[2] * r
        + beta[3] * c**3
        + beta[4] * r**2
        + beta[5] * r * c**2
    )

def design_template(data):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return np.column_stack([
        np.ones_like(r),
        c,
        r,
        c**3,
        r**2,
        r * c**2,
    ])

def fit_template(sub, alpha=1e-6):
    X = design_template(sub)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    pred = X @ beta
    stats = {"n": len(sub), "template_rmse": math.sqrt(mean_squared_error(y, pred))}
    for name, coef in zip(TERM_NAMES, beta):
        stats[f"coef_{name}"] = float(coef)
    return beta, pred, stats

def predict_with_beta(sub, beta):
    return design_template(sub) @ np.asarray(beta, dtype=float)

def trajectory_gap(sub, beta_ref, beta_cmp, n_r0=15):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 40)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def integrate(beta, r0):
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(cgrid) - 1):
            c = cgrid[i]
            dc = float(cgrid[i + 1] - cgrid[i])
            g = float(np.clip(governing_field(r, c, beta), -flow_cap, flow_cap))
            r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        return np.array(traj)

    return float(np.mean([
        math.sqrt(mean_squared_error(integrate(beta_ref, r0), integrate(beta_cmp, r0)))
        for r0 in r0s
    ]))

print("Locked field:")
print("g(r,c;β) = β0 + βc c + βr r + βc3 c^3 + βr2 r^2 + βrc2 r c^2")

## 3. Coefficient extraction

Each regime receives one coefficient vector β by fitting the shared field template.

In [ ]:
group_cols = ["system", "task", "forcing_mode", "k", "flow_mode"]
regime_rows = []
regime_subsets = {}

for vals, sub in df.groupby(group_cols):
    if len(sub) < 30:
        continue

    beta, pred, stats = fit_template(sub.copy())
    kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
    regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"

    row = {
        "regime_id": regime_id,
        "system": vals[0],
        "task": vals[1],
        "forcing_mode": vals[2],
        "k": float(vals[3]),
        "flow_mode": vals[4],
    }
    row.update(stats)

    regime_rows.append(row)
    regime_subsets[regime_id] = sub.copy()

coef_df = pd.DataFrame(regime_rows).reset_index(drop=True)
coef_df.to_csv(os.path.join(OUTPUT_DIR, "coefficient_table.csv"), index=False)

print("Regime count:", len(coef_df))
display(coef_df.head())

## 4. Final symbolic chart

The final symbolic chart is learned with a readable feature basis, then pruned using term stability.

In [ ]:
def build_symbolic_features(df_in, columns=None, reduced_terms=None):
    X = pd.get_dummies(df_in[["forcing_mode", "flow_mode", "system", "task"]], drop_first=False)

    X["k"] = df_in["k"].astype(float).values
    X["k2"] = df_in["k"].astype(float).values ** 2

    ff = df_in["forcing_mode"].astype(str) + "__x__" + df_in["flow_mode"].astype(str)
    X_ff = pd.get_dummies(ff, prefix="ff")

    sf = df_in["system"].astype(str) + "__x__" + df_in["forcing_mode"].astype(str)
    X_sf = pd.get_dummies(sf, prefix="sf")

    X_fk = pd.get_dummies(df_in["forcing_mode"].astype(str), prefix="fk").multiply(
        df_in["k"].astype(float).to_numpy().reshape(-1, 1)
    )

    X = pd.concat([X, X_ff, X_sf, X_fk], axis=1)

    if reduced_terms is not None:
        X = X.reindex(columns=reduced_terms, fill_value=0.0)

    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0.0)

    return X.astype(float)

def format_equation(name, term_list, intercept):
    parts = [f"{name} = {intercept:.5f}"]
    for feat, val in term_list:
        sign = "+" if val >= 0 else "-"
        parts.append(f"{sign} {abs(val):.5f}·{feat}")
    return " ".join(parts)

def term_stability_table(df_in, n_splits=8, threshold=1e-4):
    n = len(df_in)
    splitter = LeaveOneOut() if n <= 12 else KFold(n_splits=min(n_splits, n), shuffle=True, random_state=42)

    all_features = build_symbolic_features(df_in).columns.tolist()
    stability = {coef: {feat: 0 for feat in all_features} for coef in COEF_COLS}
    fold_count = 0

    for train_idx, _ in splitter.split(df_in):
        train_df = df_in.iloc[train_idx].reset_index(drop=True)
        X_train = build_symbolic_features(train_df, columns=all_features)

        for coef in COEF_COLS:
            y = train_df[coef].to_numpy(dtype=float)
            scaler = StandardScaler()
            Xs = scaler.fit_transform(X_train)
            model = LassoCV(cv=min(5, len(train_df)), random_state=fold_count + 1, max_iter=30000)
            model.fit(Xs, y)

            for feat, val in zip(all_features, model.coef_):
                if abs(val) > threshold:
                    stability[coef][feat] += 1

        fold_count += 1

    rows = []
    for coef in COEF_COLS:
        for feat in all_features:
            rows.append({
                "coefficient": coef,
                "term": feat,
                "frequency": stability[coef][feat] / fold_count,
                "count": stability[coef][feat],
                "folds": fold_count,
            })
    return pd.DataFrame(rows)

stability_df = term_stability_table(coef_df)
stability_df.to_csv(os.path.join(OUTPUT_DIR, "term_stability.csv"), index=False)

STABILITY_THRESHOLD = 0.5
stable_terms_by_coef = {}

for coef in COEF_COLS:
    sub = stability_df[
        (stability_df["coefficient"] == coef)
        & (stability_df["frequency"] >= STABILITY_THRESHOLD)
    ]
    stable_terms_by_coef[coef] = sub["term"].tolist()

stable_terms_table = pd.DataFrame([
    {
        "coefficient": coef,
        "n_stable_terms": len(stable_terms_by_coef[coef]),
        "stable_terms": ", ".join(stable_terms_by_coef[coef]),
    }
    for coef in COEF_COLS
])

stable_terms_table.to_csv(os.path.join(OUTPUT_DIR, "stable_terms_by_coefficient.csv"), index=False)
display(stable_terms_table)

In [ ]:
def fit_pruned_chart(df_in, terms_by_coef):
    equations = []
    preds = np.zeros((len(df_in), len(COEF_COLS)), dtype=float)
    fitted = {}

    for j, coef in enumerate(COEF_COLS):
        terms = terms_by_coef.get(coef, [])
        y = df_in[coef].to_numpy(dtype=float)

        if len(terms) == 0:
            intercept = float(np.mean(y))
            preds[:, j] = intercept
            equations.append({"coefficient": coef, "n_terms": 0, "equation": f"{coef} = {intercept:.5f}"})
            fitted[coef] = {"type": "constant", "value": intercept, "terms": []}
            continue

        X = build_symbolic_features(df_in, reduced_terms=terms)
        scaler = StandardScaler()
        Xs = scaler.fit_transform(X)
        model = Ridge(alpha=1.0)
        model.fit(Xs, y)
        preds[:, j] = model.predict(Xs)

        active_terms = [(feat, val) for feat, val in zip(X.columns, model.coef_) if abs(val) > 1e-6]
        equations.append({
            "coefficient": coef,
            "n_terms": len(active_terms),
            "equation": format_equation(coef, active_terms, model.intercept_),
        })
        fitted[coef] = {
            "type": "ridge_standardized",
            "model": model,
            "scaler": scaler,
            "terms": X.columns.tolist(),
        }

    return pd.DataFrame(equations), preds, fitted

final_equations_df, pruned_fit_preds, final_chart = fit_pruned_chart(coef_df, stable_terms_by_coef)
final_equations_df.to_csv(os.path.join(OUTPUT_DIR, "final_pruned_symbolic_equations.csv"), index=False)

display(final_equations_df)
for eq in final_equations_df["equation"]:
    print(eq)
    print()

### Optional executable symbolic chart

This function uses the fitted final pruned chart to generate coefficients directly from metadata rows.

In [ ]:
def symbolic_chart(meta_df):
    meta_df = pd.DataFrame(meta_df).copy()
    Yhat = np.zeros((len(meta_df), len(COEF_COLS)), dtype=float)

    for j, coef in enumerate(COEF_COLS):
        pack = final_chart[coef]

        if pack["type"] == "constant":
            Yhat[:, j] = pack["value"]
            continue

        X = build_symbolic_features(meta_df, reduced_terms=pack["terms"])
        Xs = pack["scaler"].transform(X)
        Yhat[:, j] = pack["model"].predict(Xs)

    return pd.DataFrame(Yhat, columns=COEF_COLS)

display(symbolic_chart(coef_df.head()))

## 5. Paper Figure 1 — Coefficient manifold

Purpose: show that coefficient vectors form a low-dimensional structured manifold.

In [ ]:
coef_scaler = StandardScaler()
Y = coef_df[COEF_COLS].to_numpy(dtype=float)
Ystd = coef_scaler.fit_transform(Y)

pca = PCA(n_components=min(6, len(COEF_COLS)), random_state=42)
Z = pca.fit_transform(Ystd)

coef_df["pc1"] = Z[:, 0]
coef_df["pc2"] = Z[:, 1]
coef_df["pc3"] = Z[:, 2] if Z.shape[1] > 2 else 0.0

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["forcing_mode", "flow_mode", "system"]):
    for val in sorted(coef_df[col].astype(str).unique()):
        sub = coef_df[coef_df[col].astype(str) == val]
        ax.scatter(sub["pc1"], sub["pc2"], label=val, s=45)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(f"Coefficient manifold by {col}")
    ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure1_coefficient_manifold.png"), dpi=220, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(5.5, 4))
ax.plot(np.arange(1, len(pca.explained_variance_ratio_) + 1), np.cumsum(pca.explained_variance_ratio_), marker="o")
ax.set_xlabel("component")
ax.set_ylabel("cumulative explained variance")
ax.set_title("Coefficient manifold cumulative variance")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure1b_cumulative_variance.png"), dpi=220, bbox_inches="tight")
plt.show()

pca_summary = pd.DataFrame({
    "component": np.arange(1, len(pca.explained_variance_ratio_) + 1),
    "explained_variance": pca.explained_variance_ratio_,
    "cumulative_explained_variance": np.cumsum(pca.explained_variance_ratio_),
})
display(pca_summary)
pca_summary.to_csv(os.path.join(OUTPUT_DIR, "pca_summary.csv"), index=False)

## 6. Paper Figure 2 — Symbolic term stability

Purpose: justify pruning and show stable symbolic structure.

In [ ]:
stable_pivot = stability_df.pivot(index="coefficient", columns="term", values="frequency").fillna(0.0)

fig, ax = plt.subplots(figsize=(max(12, 0.45 * stable_pivot.shape[1]), 4))
im = ax.imshow(stable_pivot.values, aspect="auto", vmin=0, vmax=1)
ax.set_yticks(range(len(stable_pivot.index)))
ax.set_yticklabels(stable_pivot.index)
ax.set_xticks(range(len(stable_pivot.columns)))
ax.set_xticklabels(stable_pivot.columns, rotation=45, ha="right")
ax.set_title("Symbolic term stability across folds")
fig.colorbar(im, ax=ax, label="selection frequency")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure2_term_stability.png"), dpi=220, bbox_inches="tight")
plt.show()

## 7. Paper Figure 3 — Universality boundary

Purpose: compare the final pruned symbolic chart against full symbolic and ridge baselines under harder holdouts.

In [ ]:
def predict_full_symbolic(train_df, test_df):
    X_train = build_symbolic_features(train_df)
    X_test = build_symbolic_features(test_df, columns=X_train.columns)

    Yhat = np.zeros((len(test_df), len(COEF_COLS)), dtype=float)

    for j, coef in enumerate(COEF_COLS):
        y_train = train_df[coef].to_numpy(dtype=float)
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X_train)
        Xte = scaler.transform(X_test)
        model = LassoCV(cv=min(5, len(train_df)), random_state=42, max_iter=30000)
        model.fit(Xtr, y_train)
        Yhat[:, j] = model.predict(Xte)

    return Yhat

def predict_pruned_symbolic(train_df, test_df, terms_by_coef):
    Yhat = np.zeros((len(test_df), len(COEF_COLS)), dtype=float)

    for j, coef in enumerate(COEF_COLS):
        terms = terms_by_coef.get(coef, [])

        if len(terms) == 0:
            Yhat[:, j] = train_df[coef].mean()
            continue

        X_train = build_symbolic_features(train_df, reduced_terms=terms)
        X_test = build_symbolic_features(test_df, columns=X_train.columns, reduced_terms=terms)
        y_train = train_df[coef].to_numpy(dtype=float)

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X_train)
        Xte = scaler.transform(X_test)

        model = Ridge(alpha=1.0)
        model.fit(Xtr, y_train)
        Yhat[:, j] = model.predict(Xte)

    return Yhat

def predict_ridge_chart(train_df, test_df):
    X_train = build_symbolic_features(train_df)
    X_test = build_symbolic_features(test_df, columns=X_train.columns)
    model = Ridge(alpha=1.0)
    model.fit(X_train, train_df[COEF_COLS].to_numpy(dtype=float))
    return model.predict(X_test)

hard_block_masks = {
    "k_high": coef_df["k"].astype(float) == 7.0,
    "k_low": coef_df["k"].astype(float) == 3.0,
    "forcing::condition": coef_df["forcing_mode"].astype(str) == "condition_gap",
    "forcing::capacity": coef_df["forcing_mode"].astype(str) == "capacity_gap",
    "forcing::feature": coef_df["forcing_mode"].astype(str) == "feature_gap",
    "system::entropy": coef_df["system"].astype(str) == "entropy",
    "system::unevenness": coef_df["system"].astype(str) == "unevenness",
    "flow::linear": coef_df["flow_mode"].astype(str) == "linear",
    "flow::nonlinear": coef_df["flow_mode"].astype(str) == "nonlinear",
}

rows = []

for block_name, test_mask in hard_block_masks.items():
    train_df = coef_df.loc[~test_mask].reset_index(drop=True)
    test_df = coef_df.loc[test_mask].reset_index(drop=True)

    if len(test_df) == 0 or len(train_df) < 5:
        continue

    predictions = {
        "full_symbolic": predict_full_symbolic(train_df, test_df),
        "pruned_symbolic": predict_pruned_symbolic(train_df, test_df, stable_terms_by_coef),
        "ridge_chart": predict_ridge_chart(train_df, test_df),
    }

    for i, rid in enumerate(test_df["regime_id"].tolist()):
        beta_true = test_df.loc[i, COEF_COLS].to_numpy(dtype=float)
        sub = regime_subsets[rid]
        y_emp = sub["predicted_flow"].to_numpy(dtype=float)

        for model_name, Yhat in predictions.items():
            beta_hat = Yhat[i]
            pred_field = predict_with_beta(sub, beta_hat)
            rows.append({
                "block": block_name,
                "regime_id": rid,
                "model": model_name,
                "coef_rmse": math.sqrt(mean_squared_error(beta_true, beta_hat)),
                "field_rmse": math.sqrt(mean_squared_error(y_emp, pred_field)),
                "traj_rmse": trajectory_gap(sub, beta_true, beta_hat),
            })

eval_df = pd.DataFrame(rows)
summary_df = eval_df.groupby(["block", "model"])[["coef_rmse", "field_rmse", "traj_rmse"]].mean().reset_index()
summary_df.to_csv(os.path.join(OUTPUT_DIR, "universality_boundary_summary.csv"), index=False)

display(summary_df.sort_values(["block", "traj_rmse"]))

In [ ]:
for metric in ["field_rmse", "traj_rmse"]:
    pivot = summary_df.pivot(index="block", columns="model", values=metric)
    fig, ax = plt.subplots(figsize=(12, 4.8))
    pivot.plot(kind="bar", ax=ax)
    ax.set_title(f"Universality boundary — {metric}")
    ax.set_ylabel(metric)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f"figure3_universality_{metric}.png"), dpi=220, bbox_inches="tight")
    plt.show()

## 8. Paper Figure 4 — Representative reconstruction

Purpose: show actual reconstructed residual-flow fields, not only aggregate metrics.

In [ ]:
# choose a hard block where pruned symbolic is good relative to alternatives
candidate = eval_df[
    (eval_df["model"] == "pruned_symbolic")
    & (eval_df["block"].astype(str).str.contains("forcing::condition"))
].copy()

if len(candidate) == 0:
    candidate = eval_df[eval_df["model"] == "pruned_symbolic"].copy()

rep_rid = candidate.sort_values("traj_rmse").iloc[0]["regime_id"]

train_df = coef_df[coef_df["regime_id"] != rep_rid].reset_index(drop=True)
test_df = coef_df[coef_df["regime_id"] == rep_rid].reset_index(drop=True)

beta_true = test_df[COEF_COLS].to_numpy(dtype=float)[0]
beta_pruned = predict_pruned_symbolic(train_df, test_df, stable_terms_by_coef)[0]
beta_ridge = predict_ridge_chart(train_df, test_df)[0]

sub = regime_subsets[rep_rid]
y_emp = sub["predicted_flow"].to_numpy(dtype=float)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
panels = [
    ("Empirical", y_emp),
    ("Pruned symbolic", predict_with_beta(sub, beta_pruned)),
    ("Ridge chart", predict_with_beta(sub, beta_ridge)),
]

for ax, (title, vals) in zip(axes, panels):
    sc = ax.scatter(sub["condition_coord"], sub["residual"], c=vals, s=16)
    ax.set_title(title)
    ax.set_xlabel("condition coordinate c")
    plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)

axes[0].set_ylabel("residual r")
fig.suptitle(f"Representative field reconstruction: {rep_rid}", y=1.03)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure4_representative_reconstruction.png"), dpi=220, bbox_inches="tight")
plt.show()

pd.DataFrame({
    "term": COEF_COLS,
    "true": beta_true,
    "pruned_symbolic": beta_pruned,
    "ridge_chart": beta_ridge,
}).to_csv(os.path.join(OUTPUT_DIR, "representative_coefficients.csv"), index=False)

## 9. Paper Figure 5 — Minimal chart tradeoff

Purpose: show how pruning changes generalization error.

In [ ]:
threshold_grid = [0.0, 0.125, 0.25, 0.375, 0.5, 0.625, 0.75]
threshold_rows = []

for threshold in threshold_grid:
    terms_by_coef = {}
    for coef in COEF_COLS:
        sub = stability_df[
            (stability_df["coefficient"] == coef)
            & (stability_df["frequency"] >= threshold)
        ]
        terms_by_coef[coef] = sub["term"].tolist()

    n_terms_total = sum(len(v) for v in terms_by_coef.values())
    block_scores = []

    for block_name, test_mask in hard_block_masks.items():
        train_df = coef_df.loc[~test_mask].reset_index(drop=True)
        test_df = coef_df.loc[test_mask].reset_index(drop=True)

        if len(test_df) == 0 or len(train_df) < 5:
            continue

        Yhat = predict_pruned_symbolic(train_df, test_df, terms_by_coef)

        for i, rid in enumerate(test_df["regime_id"].tolist()):
            beta_true = test_df.loc[i, COEF_COLS].to_numpy(dtype=float)
            sub = regime_subsets[rid]
            y_emp = sub["predicted_flow"].to_numpy(dtype=float)
            pred_field = predict_with_beta(sub, Yhat[i])
            block_scores.append({
                "block": block_name,
                "coef_rmse": math.sqrt(mean_squared_error(beta_true, Yhat[i])),
                "field_rmse": math.sqrt(mean_squared_error(y_emp, pred_field)),
                "traj_rmse": trajectory_gap(sub, beta_true, Yhat[i]),
            })

    score_df = pd.DataFrame(block_scores)
    threshold_rows.append({
        "stability_threshold": threshold,
        "n_terms_total": n_terms_total,
        "mean_coef_rmse": score_df["coef_rmse"].mean(),
        "mean_field_rmse": score_df["field_rmse"].mean(),
        "mean_traj_rmse": score_df["traj_rmse"].mean(),
    })

tradeoff_df = pd.DataFrame(threshold_rows)
tradeoff_df.to_csv(os.path.join(OUTPUT_DIR, "minimal_chart_tradeoff.csv"), index=False)
display(tradeoff_df)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(tradeoff_df["n_terms_total"], tradeoff_df["mean_traj_rmse"], marker="o")
for _, row in tradeoff_df.iterrows():
    ax.text(row["n_terms_total"], row["mean_traj_rmse"], f"{row['stability_threshold']:.2f}", fontsize=8)
ax.set_xlabel("total symbolic terms")
ax.set_ylabel("mean trajectory RMSE")
ax.set_title("Minimal chart tradeoff")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure5_minimal_chart_tradeoff.png"), dpi=220, bbox_inches="tight")
plt.show()

## 10. Statistical validation: within vs between cluster variance

This diagnostic checks whether visual separation in coefficient space is supported by variance ratios and silhouette scores.

In [ ]:
def variance_ratio_by_label(label_col):
    Y = coef_df[COEF_COLS].to_numpy(dtype=float)
    labels = coef_df[label_col].astype(str).to_numpy()

    global_mean = Y.mean(axis=0)
    total_ss = np.sum((Y - global_mean) ** 2)

    within_ss = 0.0
    between_ss = 0.0

    for label in sorted(set(labels)):
        sub = Y[labels == label]
        mean = sub.mean(axis=0)
        within_ss += np.sum((sub - mean) ** 2)
        between_ss += len(sub) * np.sum((mean - global_mean) ** 2)

    ratio = between_ss / max(within_ss, 1e-12)
    explained = between_ss / max(total_ss, 1e-12)

    return {
        "label": label_col,
        "within_ss": within_ss,
        "between_ss": between_ss,
        "between_within_ratio": ratio,
        "between_fraction_total": explained,
    }

validation_rows = []
for label_col in ["forcing_mode", "flow_mode", "system", "task"]:
    row = variance_ratio_by_label(label_col)

    labels = coef_df[label_col].astype(str)
    if labels.nunique() > 1 and len(coef_df) > labels.nunique():
        try:
            row["silhouette_pc2"] = silhouette_score(coef_df[["pc1", "pc2"]].to_numpy(), labels)
            row["silhouette_coef"] = silhouette_score(StandardScaler().fit_transform(coef_df[COEF_COLS]), labels)
        except Exception:
            row["silhouette_pc2"] = np.nan
            row["silhouette_coef"] = np.nan
    else:
        row["silhouette_pc2"] = np.nan
        row["silhouette_coef"] = np.nan

    validation_rows.append(row)

validation_df = pd.DataFrame(validation_rows)
validation_df.to_csv(os.path.join(OUTPUT_DIR, "cluster_validation.csv"), index=False)
display(validation_df)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.bar(validation_df["label"], validation_df["between_within_ratio"])
ax.set_ylabel("between / within variance ratio")
ax.set_title("Regime partition strength")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure6_partition_strength.png"), dpi=220, bbox_inches="tight")
plt.show()

## 11. Final model summary table

In [ ]:
final_summary = summary_df.groupby("model")[["coef_rmse", "field_rmse", "traj_rmse"]].mean().reset_index()
final_summary = final_summary.sort_values("traj_rmse")
final_summary.to_csv(os.path.join(OUTPUT_DIR, "final_model_summary.csv"), index=False)

display(final_summary)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric in zip(axes, ["coef_rmse", "field_rmse", "traj_rmse"]):
    ax.bar(final_summary["model"], final_summary[metric])
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure7_final_model_summary.png"), dpi=220, bbox_inches="tight")
plt.show()

## 12. Final verdict table

In [ ]:
decision_rows = []

for block, sub in summary_df.groupby("block"):
    best = sub.sort_values("traj_rmse").iloc[0]
    pruned = sub[sub["model"] == "pruned_symbolic"].iloc[0]
    full = sub[sub["model"] == "full_symbolic"].iloc[0]
    ridge = sub[sub["model"] == "ridge_chart"].iloc[0]
    pruned_to_best = pruned["traj_rmse"] / max(best["traj_rmse"], 1e-12)

    if best["model"] == "pruned_symbolic":
        verdict = "minimal symbolic chart wins"
    elif pruned_to_best <= 1.25:
        verdict = "minimal symbolic chart competitive"
    elif best["model"] == "full_symbolic":
        verdict = "full symbolic chart needed"
    else:
        verdict = "regularized dense chart wins"

    decision_rows.append({
        "block": block,
        "best_model": best["model"],
        "best_traj_rmse": best["traj_rmse"],
        "pruned_traj_rmse": pruned["traj_rmse"],
        "full_traj_rmse": full["traj_rmse"],
        "ridge_traj_rmse": ridge["traj_rmse"],
        "pruned_to_best_ratio": pruned_to_best,
        "verdict": verdict,
    })

decision_df = pd.DataFrame(decision_rows)
decision_df.to_csv(os.path.join(OUTPUT_DIR, "final_verdict_table.csv"), index=False)
display(decision_df)

## 13. Paper-ready abstract and conclusion draft

In [ ]:
abstract = f'''
# Draft Abstract

We study residual-flow dynamics across zeta-constraint regimes and show that the observed dynamics admit a two-layer representation. First, residual evolution in coordinates `(r,c)` is captured by a shared low-order governing field,

`g(r,c;β) = β0 + βc c + βr r + βc3 c^3 + βr2 r^2 + βrc2 r c^2`.

Second, regime variation is concentrated in the coefficient vector `β`, which lies on a low-dimensional manifold: the first three principal components explain {np.cumsum(pca.explained_variance_ratio_)[2]:.3f} of standardized coefficient variance. Rather than requiring smooth latent ODE transport, the coefficient manifold is best represented by a sparse symbolic coordinate chart conditioned on forcing mode, flow mode, system, task, and `k`. Term-stability analysis identifies a minimal chart with {sum(len(v) for v in stable_terms_by_coef.values())} active stable terms. Holdout evaluations expose a universality boundary: sparse symbolic charts remain competitive or superior under harder forcing and k extrapolation blocks, while dense ridge baselines can fail. These results support a shared-field plus symbolic-chart representation for residual-flow generalization.
'''

conclusion = '''
# Draft Conclusion

The zeta-constraint residual-flow pipeline closes around a simple mathematical structure. A shared low-order field governs local residual dynamics, while regime dependence appears as a sparse symbolic chart over the field coefficients. The coefficient manifold is low-dimensional and partitioned by forcing, flow, and system variables. Attempts to model coefficient change as affine or quadratic latent ODE transport are useful as diagnostics, but do not replace the direct symbolic coordinate chart. The final abstraction is therefore not a smooth transport law in latent space but a chart-based mapping from regime variables to governing-field coefficients. The universality boundary is visible in harder holdout blocks: sparse symbolic structure generalizes where fuller or denser models overfit.
'''

with open(os.path.join(OUTPUT_DIR, "abstract.md"), "w", encoding="utf-8") as f:
    f.write(abstract.strip() + "\n")

with open(os.path.join(OUTPUT_DIR, "conclusion.md"), "w", encoding="utf-8") as f:
    f.write(conclusion.strip() + "\n")

display(Markdown(abstract))
display(Markdown(conclusion))

## 14. Export manifest

In [ ]:
manifest = {
    "data_source": data_source,
    "synthetic_fallback": bool(USE_SYNTHETIC_FALLBACK),
    "regime_count": int(len(coef_df)),
    "coefficient_terms": COEF_COLS,
    "pca_explained_variance": pca.explained_variance_ratio_.tolist(),
    "pca_cumulative_explained_variance": np.cumsum(pca.explained_variance_ratio_).tolist(),
    "stability_threshold": STABILITY_THRESHOLD,
    "total_stable_terms": int(sum(len(v) for v in stable_terms_by_coef.values())),
    "outputs": sorted(os.listdir(OUTPUT_DIR)),
}

with open(os.path.join(OUTPUT_DIR, "manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

display(pd.DataFrame({"output_file": sorted(os.listdir(OUTPUT_DIR))}))
print("Saved all paper artifacts under:", OUTPUT_DIR)

## Final Statement

The final result is:

\[
\text{data}
\rightarrow
\beta
\rightarrow
\text{coefficient manifold}
\rightarrow
\text{sparse symbolic chart}
\rightarrow
g(r,c;\beta)
\]

The important negative result is also retained:

> Smooth latent ODE transport was tested and rejected as the final abstraction.

The paper-ready claim is:

> A shared governing residual field exists, and its regime variation is captured by a low-dimensional sparse symbolic coordinate chart. Generalization boundaries emerge when regimes leave the symbolic chart support, but pruning stabilizes the chart against dense-model failure.

Recommended next step:

**Notebook 65 — LaTeX paper assembly from Notebook 64 artifacts**